<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2);">
  <tr>
    <td>
      <h1 style="margin-bottom: 0; color: black; font-size: clamp(1.5rem, 2.5vw, 2.5rem);">
        Drone RL with LLM-Guided Curriculum Learning
      </h1>
      <p style="margin-top: 8px; color: black; font-size: 1rem;">
        EVA Deep Reinforcement Learning &ndash; DRL Project, FS 2026<br>
        Rino M. Albertin  
      </p>
    </td>
    <td align="right">
      <img src="docs/images/OST_Logo_DE_RGB@2000ppi.png" alt="OST Logo" width="180">
    </td>
  </tr>
</table>

## Introduction

This project investigates single-drone trajectory tracking in simulation. A PPO agent is trained in a `gym-pybullet-drones` / PyBullet environment to follow reference trajectories.

The comparison is between direct PPO training, a fixed manual curriculum, and a local-LLM-guided curriculum. The LLM proposes structured task descriptions only; deterministic validation and repair logic decide whether a proposed task can be used. The LLM does not control the drone and does not generate executable Python code.

## Objective and Approach

The goal is to compare how different training setups affect trajectory tracking:

| Group | Purpose | Action interfaces |
|---|---|---|
| Direct PPO baselines | Train directly on `basic_training_show` or `tracking_medium` distributions. | `pid_position`, `direct_rpm` |
| Manual curriculum | Train through fixed stages from hover to medium tracking. | `pid_position`, `direct_rpm` |
| LLM curriculum | Use deterministic hover bootstrap, then validated JSON task proposals from a local LLM. | `pid_position`, `direct_rpm` |
| PPO variants | Compare architecture and hyperparameter profiles on `tracking_medium`. | `pid_position`, `direct_rpm` |

The planned experiment matrix contains 18 runs. The execution scripts organize these runs operationally, but the report focuses on the method comparison rather than the execution partitioning.

## Environment and Agent

The tracking environment wraps `HoverAviary` with a compact Gymnasium interface. The base observation contains current position, reference position, position error, and normalized trajectory progress. The active training configs also enable dynamics observation and previous action, adding velocity, attitude, angular velocity, and the previous PPO-facing action.

The two action interfaces are:

| Interface | Description |
|---|---|
| `pid_position` | PPO outputs normalized target-position commands that are mapped to PID target positions. This is the higher-level control interface. |
| `direct_rpm` | PPO outputs normalized per-motor commands that are mapped to motor RPMs before PyBullet physics. This is more direct and harder to stabilize. |

`tracking_medium` is a task distribution, not one fixed trajectory. During training, a task can be sampled on each reset. The sampled task then remains constant within that episode. Own-task evaluation uses a deterministic representative task, while generalization and scenario evaluations are separate tests.

## Algorithm

The project uses Proximal Policy Optimization (PPO) from Stable-Baselines3. PPO is a policy-optimization method, not Q-learning. It updates an actor policy while using a critic/value function to estimate returns and advantages.

The active configs use `MlpPolicy`. The default policy network is `net128`, with actor and critic networks `[128, 128]`. The `net256` variant uses `[256, 256]`. Scheduled training configs use 500,000 timesteps, 4 vectorized environments, 240 evaluation steps, seed 0, dynamics observation, and previous-action observation.

Compared PPO profiles include the default settings, `net256`, low learning rate (`0.0001`), entropy coefficient `0.005`, clip range `0.10`, and target KL `0.015`.

## Training and Test / Evaluation

Training is run from config files rather than from this notebook. The most relevant configs are:

| Purpose | Configs |
|---|---|
| Direct PPO | `configs/training/ppo_tracking_*` |
| Manual curriculum | `configs/curricula/curriculum_*_m-taskdist_medium.yaml` |
| LLM curriculum | `configs/curricula/llm_curriculum_*_m-taskdist_medium.yaml` |
| Generalization evaluation | `configs/evaluation/generalization_eval_suite.yaml` |
| Scenario/show evaluation | `configs/evaluation/scenario_task_catalog.yaml` |

Evaluation separates three questions:

1. Own-task evaluation: how well the final policy tracks its deterministic representative task.
2. Generalization evaluation: how well the policy handles fixed tasks outside the sampled training episode.
3. Scenario/show evaluation: how the policy behaves in curated visual scenarios such as easy, medium, or hard show cases.

No training is launched from this notebook.

### Available Artifacts

Generated artifacts are expected directly under:

```text
/workspace/storage/runs/<run_name>/
```

At notebook preparation time, no representative plots, GIFs, manifests, or metrics were available under `/workspace/storage/runs/`. The table below therefore records the intended representative artifacts and their current status.

| Representative artifact | Run or scenario | Status |
|---|---|---|
| Direct PPO PID tracking-medium plot/GIF | `direct_ppo_pid_dynprev_m-taskdist_medium_seed0` | Not available under `/workspace/storage/runs/` |
| Direct PPO direct-RPM tracking-medium plot/GIF | `direct_ppo_directrpm_dynprev_m-taskdist_medium_seed0` | Not available under `/workspace/storage/runs/` |
| Manual curriculum plot/GIF | `curriculum_manual_pid_dynprev_m-taskdist_medium_seed0` or direct-RPM counterpart | Not available under `/workspace/storage/runs/` |
| LLM curriculum plot/GIF | `llm_curriculum_pid_dynprev_m-taskdist_medium_seed0` or direct-RPM counterpart | Not available under `/workspace/storage/runs/` |
| Scenario/show medium GIF | `show_medium` evaluation artifact | Not available under `/workspace/storage/runs/` |

If these files are generated later, they should be embedded directly with Markdown image links rather than hidden behind notebook helper code.

### Metric Summary

The code cell below only reads existing metrics JSON files from `/workspace/storage/runs/`. It does not start training or evaluation. If metrics are missing, it prints an explicit TODO-style placeholder.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown
from IPython.display import display as ipy_display

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

evaluation = importlib.import_module("src.evaluation")

ARTIFACT_ROOT = Path("/workspace/storage/runs")
comparison_rows = evaluation.report.build_metric_comparison_table(root=ARTIFACT_ROOT)

if comparison_rows:
    comparison_table = pd.DataFrame(comparison_rows)
    columns = [
        "run_name",
        "method",
        "action_interface",
        "training_target",
        "ppo_variant",
        "evaluation_name",
        "mean_tracking_error",
        "final_error",
        "max_error",
        "termination_status",
        "metric_file",
    ]
    ipy_display(comparison_table.loc[:, columns].head(20))
else:
    ipy_display(Markdown("_No metrics JSON files are available under `/workspace/storage/runs/`; final numeric comparison is TODO._"))

## Conclusion and Outlook

The repository implements PPO trajectory tracking experiments for direct training, fixed manual curricula, and local-LLM-guided curricula. The active experiment matrix covers both `pid_position` and `direct_rpm`, plus PPO architecture and hyperparameter variants.

The implementation and evaluation pipeline are in place, but final numeric conclusions require generated metrics and representative plots/GIFs under `/workspace/storage/runs/`. Without those artifacts, this notebook reports the setup and marks results as not available rather than inventing values.

Useful future work includes more random seeds, longer training, stronger baselines, SAC or TD3 comparisons, improved direct-RPM stabilization, broader generalization tests, and improved LLM proposal strategy.

## References

- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., and Klimov, O. "Proximal Policy Optimization Algorithms." arXiv:1707.06347, 2017.
- Raffin, A., Hill, A., Gleave, A., Kanervisto, A., Ernestus, M., and Dormann, N. "Stable-Baselines3: Reliable Reinforcement Learning Implementations." Journal of Machine Learning Research, 2021.
- Panerati, J., Zheng, H., Zhou, S., Xu, J., Prorok, A., and Schoellig, A. P. "Learning to Fly: A Gym Environment with PyBullet Physics for Reinforcement Learning of Multi-agent Quadcopter Control." 2021.
- Coumans, E. and Bai, Y. PyBullet, a Python module for physics simulation.
- Farama Foundation. Gymnasium documentation and API.

*The AI tool* **ChatGPT** *from OpenAI (GPT-5.5, https://chatgpt.com) was used for linguistic editing and assistance with code snippets.  
The author bears full responsibility for the technical accuracy and content.*

## Declaration of Independence
I hereby certify that I have written this thesis independently and have not used any auxiliary materials other than those indicated. The passages of the work, which are taken in the wording or the sense after other works (to it also Internet sources count), were marked under indication of the source.

<table style="width:100%; background-color: white; padding: 10px; border-radius: 6px; box-shadow: 0 0 5px rgba(0,0,0,0.2); margin-top:20px;">
  <tr>
    <td align="left">
      <img src="docs/images/Unterschrift.png" alt="Unterschrift" style="height:80px;">
    </td>
  </tr>
</table>